In [92]:
import torch
torch.set_printoptions(precision=4, sci_mode=False)

In [93]:
torch.manual_seed(233)
inputs = torch.randn(6,3 )
inputs

tensor([[-0.3326,  0.3805, -2.1426],
        [ 1.0135, -1.8417,  0.7673],
        [-0.2934,  1.2548, -0.3498],
        [-0.1778, -0.3024, -1.5551],
        [-0.6625,  2.1011, -1.7592],
        [ 0.8725, -0.9504, -0.7434]])

In [94]:
# let's take number 3 as query and find its score with all 
query_vec = inputs[2]
query_vec

tensor([-0.2934,  1.2548, -0.3498])

In [95]:
atten_score = torch.zeros(6, 1)
atten_score.shape, atten_score

(torch.Size([6, 1]),
 tensor([[0.],
         [0.],
         [0.],
         [0.],
         [0.],
         [0.]]))

In [96]:
for i, vec in enumerate(inputs):
    atten_score[i] = query_vec.dot(vec)
atten_score.shape, atten_score

(torch.Size([6, 1]),
 tensor([[ 1.3245],
         [-2.8767],
         [ 1.7829],
         [ 0.2167],
         [ 3.4462],
         [-1.1885]]))

In [97]:
# normalize it 
atten_score / atten_score.sum()

tensor([[ 0.4896],
        [-1.0634],
        [ 0.6591],
        [ 0.0801],
        [ 1.2739],
        [-0.4393]])

In [98]:
def custom_softmax(x):
    x = torch.exp(x) / torch.exp(x).sum()
    print(x)


custom_softmax(atten_score)

tensor([[0.0881],
        [0.0013],
        [0.1393],
        [0.0291],
        [0.7351],
        [0.0071]])


In [99]:
# but if you want a stablize, postive number's normalization use the softmax
# softmax = torch.exp(i) / sum(torch.exp(i))

atten_score = torch.softmax(atten_score, dim=0)
atten_score.shape, atten_score, atten_score.sum()

(torch.Size([6, 1]),
 tensor([[0.0881],
         [0.0013],
         [0.1393],
         [0.0291],
         [0.7351],
         [0.0071]]),
 tensor(1.))

In [100]:
# so it shows the relevance of the words based on the query vector, these attention scores
# give us the percentage we want to pick from the original vectors to create the context
# vector of the query guy
def get_context_vector(x, a):
    z_i = torch.zeros(query_vec.shape)
    print(z_i.shape, x.shape, a.shape)
    for i, _ in enumerate(x):
        # x[i] vector
        # a[i] its part we want to take
        # x[i] * a[i] -> like [1 0] * 0.5 => [0.5, 0] (that what we are taking)
        # print(z[0], x[i], a[i], x[i] * a[i])
        # z_i = x[i] * a[i]
        z_i += x[i] * a[i]
        print(z_i)
    return f"we transformed {query_vec=} to context vector={z_i}"


get_context_vector(inputs, atten_score)

torch.Size([3]) torch.Size([6, 3]) torch.Size([6, 1])
tensor([-0.0293,  0.0335, -0.1887])
tensor([-0.0280,  0.0311, -0.1877])
tensor([-0.0688,  0.2059, -0.2364])
tensor([-0.0740,  0.1971, -0.2817])
tensor([-0.5610,  1.7416, -1.5748])
tensor([-0.5548,  1.7348, -1.5801])


'we transformed query_vec=tensor([-0.2934,  1.2548, -0.3498]) to context vector=tensor([-0.5548,  1.7348, -1.5801])'

In [101]:
# Now let's do this vectorize 
inputs = torch.randn(6, 3)
inputs.shape, inputs 

(torch.Size([6, 3]),
 tensor([[-0.8734,  0.6232,  0.4526],
         [ 0.2975,  0.9239, -0.1534],
         [-0.4767,  0.9737, -2.2187],
         [ 0.4207, -1.3573,  0.6799],
         [-0.0671,  0.2659,  0.0928],
         [ 0.6899, -0.6086,  0.1529]]))

In [102]:
atten_all = inputs @ inputs.T
atten_all.shape, atten_all

(torch.Size([6, 6]),
 tensor([[ 1.3560,  0.2465,  0.0190, -0.9055,  0.2662, -0.9126],
         [ 0.2465,  0.9656,  1.0982, -1.2332,  0.2115, -0.3805],
         [ 0.0190,  1.0982,  6.0980, -3.0307,  0.0850, -1.2607],
         [-0.9055, -1.2332, -3.0307,  2.4815, -0.3260,  1.2202],
         [ 0.2662,  0.2115,  0.0850, -0.3260,  0.0838, -0.1939],
         [-0.9126, -0.3805, -1.2607,  1.2202, -0.1939,  0.8697]]))

In [103]:
atten_all = torch.softmax(atten_all, dim=-1)
atten_all, atten_all.sum(dim=-1)

(tensor([[    0.4681,     0.1543,     0.1229,     0.0488,     0.1574,     0.0484],
         [    0.1404,     0.2881,     0.3290,     0.0320,     0.1355,     0.0750],
         [    0.0023,     0.0067,     0.9879,     0.0001,     0.0024,     0.0006],
         [    0.0240,     0.0173,     0.0029,     0.7113,     0.0429,     0.2015],
         [    0.2084,     0.1973,     0.1739,     0.1153,     0.1736,     0.1315],
         [    0.0504,     0.0858,     0.0356,     0.4253,     0.1034,     0.2995]]),
 tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000]))

In [105]:
# inputs[0] += atten_all[0] *
context_vectors = torch.zeros(inputs.shape)
print(context_vectors)
print(atten_all.shape, atten_all)
print(inputs.shape, inputs)
for i in range(atten_all.shape[0]):
    # print(atten_all[i])
    for j in range(atten_all.shape[1]):
        # print(f"{atten_all[i][j]} * {inputs[j]}")
        context_vectors[i] += atten_all[i][j] * inputs[j]

context_vectors

tensor([[0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]])
torch.Size([6, 6]) tensor([[    0.4681,     0.1543,     0.1229,     0.0488,     0.1574,     0.0484],
        [    0.1404,     0.2881,     0.3290,     0.0320,     0.1355,     0.0750],
        [    0.0023,     0.0067,     0.9879,     0.0001,     0.0024,     0.0006],
        [    0.0240,     0.0173,     0.0029,     0.7113,     0.0429,     0.2015],
        [    0.2084,     0.1973,     0.1739,     0.1153,     0.1736,     0.1315],
        [    0.0504,     0.0858,     0.0356,     0.4253,     0.1034,     0.2995]])
torch.Size([6, 3]) tensor([[-0.8734,  0.6232,  0.4526],
        [ 0.2975,  0.9239, -0.1534],
        [-0.4767,  0.9737, -2.2187],
        [ 0.4207, -1.3573,  0.6799],
        [-0.0671,  0.2659,  0.0928],
        [ 0.6899, -0.6086,  0.1529]])


tensor([[-0.3782,  0.5002, -0.0295],
        [-0.1376,  0.6210, -0.6648],
        [-0.4706,  0.9696, -2.1915],
        [ 0.4181, -1.0429,  0.5203],
        [-0.0786,  0.2911, -0.2071],
        [ 0.3432, -0.5867,  0.2753]])

In [106]:
context_vectors_all = atten_all @ inputs
context_vectors_all

tensor([[-0.3782,  0.5002, -0.0295],
        [-0.1376,  0.6210, -0.6648],
        [-0.4706,  0.9696, -2.1915],
        [ 0.4181, -1.0429,  0.5203],
        [-0.0786,  0.2911, -0.2071],
        [ 0.3432, -0.5867,  0.2753]])